# 05 — Training only on informative tests?

**Question.** Does a model trained only on the informative tests (high-energy, slowly switching
heating) beat the model trained on all simulations, *on those same tests*?

* **Filter** (input only, so it is also available when measuring a real building): duty cycle
  ≥ 0.6 and mean on/off step ≥ 1.5 h. About 31 % of the simulations (median PRBS minimum step
  1.46 h vs 0.95 h overall). Filtering on the targets or on the model's errors would bias the result.
* **Models**: `final_tcn_mse_v1` (all 800k training simulations, 30 epochs; trained with the same
  pipeline as the subset model, i.e. before precise BatchNorm was added) and `subset_tcn_mse`
  (filtered training set, ~245k, 100 epochs so that both see about the same number of samples).
  Same architecture, loss and learning-rate schedule.
* **Comparison** on the filtered validation simulations (the deployment setting of the
  specialised model), and on the whole validation split for reference.

Expected: no gain (filtering removes data, not noise, since the model already sees the
excitation). A clear gain would justify a protocol-specific model.

In [1]:
import os
from pathlib import Path

import pandas as pd
import torch

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)

from ipe_prediction.training.config import DataConfig
from ipe_prediction.training.dataset import ThermalDataset, excitation_mask, make_dataloader, prepare_data
from ipe_prediction.training.engine import amp_dtype, evaluate, load_model

pd.set_option("display.precision", 3)
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RUNS = {"all simulations": "final_tcn_mse_v1", "informative only": "subset_tcn_mse"}
MIN_DUTY, MIN_STEP_HOURS = 0.6, 1.5

tensors, split = prepare_data(DataConfig())
val = split["val"]
keep = excitation_mask(tensors.heater_on[val], MIN_DUTY, MIN_STEP_HOURS)
print(f"validation: {len(val):,} simulations, {int(keep.sum()):,} pass the filter ({keep.float().mean():.1%})")

validation: 99,999 simulations, 30,747 pass the filter (30.7%)


In [2]:
rows, per_target = {}, {}
for label, run in RUNS.items():
    model, prep, cfg = load_model(ROOT / "outputs" / "runs" / run, device=str(dev))
    for subset_name, idx in [("filtered val", val[keep]), ("full val", val)]:
        loader = make_dataloader(ThermalDataset(tensors, idx), 4096)
        res = evaluate(model, prep, loader, dev, amp_dtype(dev, cfg.train.amp), cfg.train.loss)
        rows[(label, subset_name)] = {"std. MSE": res["loss"], **{
            f"MSE {n}": float(v) for n, v in zip(prep.target_names, res["loss_per_target"])}}
        if subset_name == "filtered val":
            per_target[label] = res["metrics"][["MAPE_%", "P95_APE_%", "R2"]]
display(pd.DataFrame(rows).T.round(4))
display(pd.concat(per_target, axis=1).round(3))

std. MSE  MSE HLC  MSE HLC_mit  MSE HLC_ext  \
all simulations  filtered val     0.708    0.523        0.696        0.549   
                 full val         0.738    0.582        0.732        0.590   
informative only filtered val     0.710    0.528        0.695        0.554   
                 full val         1.449    1.050        0.874        1.286   

                               MSE tau_b  MSE tau_n  MSE tau_inf  
all simulations  filtered val      0.879      0.702        0.899  
                 full val          0.879      0.742        0.902  
informative only filtered val      0.879      0.704        0.898  
                 full val          1.921      1.764        1.800

all simulations                  informative only                 
                 MAPE_% P95_APE_%     R2           MAPE_% P95_APE_%     R2
HLC               7.854    19.257  0.470            7.857    19.150  0.465
HLC_mit          15.274    40.241  0.297           15.229    39.785  0.297
HLC_ext           9.243    22.832  0.452            9.243    22.804  0.446
tau_b            14.807    36.095  0.099           14.980    36.969  0.108
tau_n            11.969    29.004  0.280           11.925    28.997  0.276
tau_inf          17.827    43.424  0.074           17.999    44.272  0.082

## Conclusion

**Training only on informative tests brings no gain.** On the 30,747 filtered validation
simulations (the specialised model's own deployment setting), the model trained on all
simulations scores a standardised MSE of **0.708** and the informative-only model **0.710**
(best epoch 79 of 100); per target the two are within ±0.01 of each other (HLC MAPE 7.85 % vs
7.86 %, HLC_ext 9.24 % vs 9.24 %, R² equal to two decimals). The difference is at the level of
run-to-run variation (about 0.001–0.002).

On the whole validation split, the specialised model collapses (1.45, worse than predicting the
mean), because it never saw unheated or fast-switching tests: it only works inside the protocol
it was trained on.

**Interpretation.** As expected, removing tests removes data, not noise. The model already sees
the excitation in its input and adapts its prediction to how informative each test is, so the
uninformative simulations do not prevent it from learning the patterns of the informative ones.
The accuracy on informative tests (0.708, HLC MAPE 7.9 %, P95 19 %) is set by what those tests
reveal, not by the training mix.

**Decision:** keep the single model trained on all simulations. It is as good on informative
tests and remains valid for every protocol.